In [ ]:
import os
from pathlib import Path


def find_repo_root():
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "downstream_tasks/expression_prediction").is_dir():
            return path
    raise RuntimeError("Run inside the GENA-LM clone or set GENA_HOME")


REPO_ROOT = (
    Path(os.environ["GENA_HOME"]).resolve()
    if "GENA_HOME" in os.environ
    else find_repo_root()
)
BENCHMARK_ROOT = Path(os.environ.get("BENCHMARK_ROOT", REPO_ROOT)).resolve()
TASK_ROOT = Path(os.environ.get("TASK_ROOT", REPO_ROOT)).resolve()
DATA_ROOT = Path(os.environ.get("DATA_ROOT", REPO_ROOT / "data")).resolve()


# GENA vs ground truth on AlphaGenome ontologies

Loop over checkpoints and splits. Results use the genes and 285 ontology groups supported by AlphaGenome.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


In [ ]:
BASE = BENCHMARK_ROOT
TRUE_PATH = DATA_ROOT / "true_mouse_all_genes_qnorm_samples_by_genes.WITH_TEST.csv"
ONTOLOGY_JSON = REPO_ROOT / "downstream_tasks/expression_prediction/benchmarks/alphagenome_benchmark/data/alphagenome_track_to_qnorm_ids_mouse_all.json"
AG_DIR = BASE / "AlphaGenome/predictions/results-seq_len1Mb_json243_mouse"
# OUTPUT_PATH = BASE / "graund_true_comparison/results/gena_ag_gt_ontology_summary.csv"

MODEL_DIRS = {
    "dev_loss": TASK_ROOT / "predictions_results_mm10/dev_loss",
    "all_datasets_2": TASK_ROOT / "predictions_results_mm10/all_datasets2",
    "all_datasets": TASK_ROOT / "predictions_results_mm10/all_datasets",
    "glioma": TASK_ROOT / "predictions_results_mm10/glioma",
    "len_2048": TASK_ROOT / "predictions_results_mm10/len_2048",
    "ATAC": TASK_ROOT / "predictions_results_mm10/ATAC",
    "mult_loss": TASK_ROOT / "predictions_results_mm10/mult_loss",
    "xlarge": TASK_ROOT / "predictions_results_mm10/xlarge",
}
MODEL_ORDER = list(MODEL_DIRS)
SPLITS = ["valid", "test"]


In [ ]:
MODEL_DIRS

In [ ]:
def read_prediction(path):
    df = pd.read_csv(path)
    if "gene_id" not in df.columns:
        df = df.rename(columns={df.columns[0]: "gene_id"})
    return df


def load_gena(model_dir, split):
    direct = model_dir / f"gena_lm_{split}_json243_mm10_predictions.csv"
    if direct.exists():
        return read_prediction(direct), "json243_mm10"

    path812 = model_dir / f"gena_lm_{split}_json243_mm10_predictions.csv"
    path3 = model_dir / f"gena_lm_{split}_json3_predictions.csv"
    if not path812.exists() or not path3.exists():
        raise FileNotFoundError(f"Need {direct.name}, or both {path812.name} and {path3.name}")

    left = read_prediction(path812).set_index("gene_id")
    right = read_prediction(path3).set_index("gene_id")
    merged = left.join(right, how="inner", validate="one_to_one").reset_index()
    return merged, "json812 + json3"


def mean_by_ontology(df, ontology_to_ids):
    x = df.set_index("gene_id")
    out = {
        ontology: x[present].astype(float).mean(axis=1)
        for ontology, ids in ontology_to_ids.items()
        if (present := [item for item in ids if item in x.columns])
    }
    return pd.DataFrame(out, index=x.index)


def _pairwise_corr(x, y, axis):
    # Pearson correlation using only positions where both arrays are finite.
    valid = np.isfinite(x) & np.isfinite(y)
    n = valid.sum(axis=axis)
    xx = np.where(valid, x, 0.0)
    yy = np.where(valid, y, 0.0)
    sx = xx.sum(axis=axis)
    sy = yy.sum(axis=axis)
    sxx = (xx * xx).sum(axis=axis)
    syy = (yy * yy).sum(axis=axis)
    sxy = (xx * yy).sum(axis=axis)
    numerator = n * sxy - sx * sy
    denominator = np.sqrt((n * sxx - sx**2) * (n * syy - sy**2))
    return np.divide(
        numerator, denominator,
        out=np.full(n.shape, np.nan, dtype=float),
        where=(n > 3) & (denominator > 0),
    )


def mean_correlations(true_df, pred_df):
    x = true_df.to_numpy(dtype=float)
    y = pred_df.to_numpy(dtype=float)
    corr_across_genes = _pairwise_corr(x, y, axis=0)
    corr_across_cells = _pairwise_corr(x, y, axis=1)
    return float(np.nanmean(corr_across_genes)), float(np.nanmean(corr_across_cells))


In [ ]:
# Ground truth: cell tracks x genes -> genes x cell tracks, then log2(x + 1).
true_raw = pd.read_csv(TRUE_PATH)
cell_id_col = "gene_id" if "gene_id" in true_raw.columns else "id"
gene_cols = [column for column in true_raw.columns if column.startswith(("ENSG", "ENSMUSG"))]
true_log = np.log2(true_raw.set_index(cell_id_col)[gene_cols].T.astype(float) + 1)
true_log.index.name = "gene_id"

with open(ONTOLOGY_JSON) as handle:
    ontology_to_ids = json.load(handle)

true_ontology = mean_by_ontology(true_log.reset_index(), ontology_to_ids)
print("GT:", true_log.shape, "->", true_ontology.shape)


In [ ]:
rows = []
ag_rows = []

for split in SPLITS:
    ag_path = AG_DIR / f"alphagenome_supported_predictions_{split}_intervals.csv"
    ag_ontology = mean_by_ontology(read_prediction(ag_path), ontology_to_ids)

    # Evaluate AlphaGenome itself against the ontology-averaged ground truth.
    ag_genes = true_ontology.index.intersection(ag_ontology.index)
    ag_ontologies = true_ontology.columns.intersection(ag_ontology.columns)
    ag_gene_corr, ag_cell_corr = mean_correlations(
        true_ontology.loc[ag_genes, ag_ontologies],
        ag_ontology.loc[ag_genes, ag_ontologies],
    )
    ag_available_per_gene = np.isfinite(ag_ontology.loc[ag_genes, ag_ontologies]).sum(axis=1)
    ag_rows.append({
        "model": "AlphaGenome", "split": split,
        "gene_corr": ag_gene_corr, "cell_corr": ag_cell_corr,
        "n_genes": len(ag_genes), "n_ontologies": len(ag_ontologies),
        "mean_ontologies_per_gene": float(ag_available_per_gene.mean()),
        "min_ontologies_per_gene": int(ag_available_per_gene.min()),
        "max_ontologies_per_gene": int(ag_available_per_gene.max()),
    })

    for checkpoint, model_dir in MODEL_DIRS.items():
        row = {"checkpoint": checkpoint, "split": split}
        try:
            gena, source = load_gena(model_dir, split)
            gena_ontology = mean_by_ontology(gena, ontology_to_ids)

            common_genes = true_ontology.index.intersection(gena_ontology.index).intersection(ag_ontology.index)
            common_ontologies = true_ontology.columns.intersection(gena_ontology.columns).intersection(ag_ontology.columns)
            if len(common_genes) == 0 or len(common_ontologies) == 0:
                raise ValueError("No common genes or ontologies")

            true_aligned = true_ontology.loc[common_genes, common_ontologies]
            gena_aligned = gena_ontology.loc[common_genes, common_ontologies]
            ag_aligned = ag_ontology.loc[common_genes, common_ontologies]

            # Fair comparison: score GENA only where AlphaGenome has a value.
            ag_available = np.isfinite(ag_aligned)
            true_for_gena = true_aligned.where(ag_available)
            gena_for_eval = gena_aligned.where(ag_available)
            gene_corr, cell_corr = mean_correlations(true_for_gena, gena_for_eval)
            available_per_gene = ag_available.sum(axis=1)
            row.update(gene_corr=gene_corr, cell_corr=cell_corr,
                       n_genes=len(common_genes), n_ontologies=len(common_ontologies),
                       mean_ontologies_per_gene=float(available_per_gene.mean()),
                       min_ontologies_per_gene=int(available_per_gene.min()),
                       max_ontologies_per_gene=int(available_per_gene.max()),
                       source=source, status="ok")
        except Exception as error:
            row.update(gene_corr=np.nan, cell_corr=np.nan,
                       n_genes=0, n_ontologies=0, source="", status=str(error))

        rows.append(row)
        print(checkpoint, split, row["status"])

results_long = pd.DataFrame(rows)
ag_results_long = pd.DataFrame(ag_rows)


In [ ]:
# Final table: valid gene_corr, cell_corr; then test gene_corr, cell_corr.
summary = results_long.pivot(
    index="checkpoint",
    columns="split",
    values=["gene_corr", "cell_corr"],
)
summary = summary.reorder_levels([1, 0], axis=1)
column_order = pd.MultiIndex.from_product(
    [["valid", "test"], ["gene_corr", "cell_corr"]],
    names=["split", "metric"],
)
summary = summary.reindex(index=MODEL_ORDER, columns=column_order)
summary.columns = pd.MultiIndex.from_tuples(
    [(f"{split} (815 cell types, 242 AG ontologies)", metric) for split, metric in summary.columns],
    names=["AlphaGenome", "metric"],
)
display(summary.round(3))


In [ ]:
# Show missing/failed runs, if any.
problems = results_long.loc[results_long["status"] != "ok", ["checkpoint", "split", "status"]]
display(problems if not problems.empty else pd.DataFrame({"status": ["All runs completed"]}))


In [ ]:
summary.round(3).to_csv("ontology.csv", float_format="%.3f")

In [ ]:
ag_results_long

In [ ]:
# AlphaGenome performance against the same ground truth.
ag_summary = ag_results_long.pivot(
    index="model",
    columns="split",
    values=["gene_corr", "cell_corr"],
).reorder_levels([1, 0], axis=1)
ag_summary = ag_summary.reindex(
    columns=pd.MultiIndex.from_product(
        [["valid", "test"], ["gene_corr", "cell_corr"]],
        names=["split", "metric"],
    )
)
display(ag_summary.round(3))
display(ag_results_long[["split", "n_genes", "n_ontologies",
                         "mean_ontologies_per_gene", "min_ontologies_per_gene",
                         "max_ontologies_per_gene"]])


In [ ]:
import json
import pandas as pd

FIRST_ID_JSON = REPO_ROOT / "downstream_tasks/expression_prediction/benchmarks/alphagenome_benchmark/data/alphagenome_track_to_qnorm_id_mouse_first.json"

ALL_IDS_JSON = REPO_ROOT / "downstream_tasks/expression_prediction/benchmarks/alphagenome_benchmark/data/alphagenome_track_to_qnorm_ids_mouse_all.json"

PREDICTION_FILE = BENCHMARK_ROOT / "AlphaGenome/predictions/results-seq_len16Kb_json243_mouse/alphagenome_supported_predictions_test_intervals.csv"

with open(FIRST_ID_JSON) as f:
    track_to_first_id = json.load(f)

with open(ALL_IDS_JSON) as f:
    track_to_all_ids = json.load(f)

pred = pd.read_csv(PREDICTION_FILE)
pred_values = pred.drop(columns="gene_id")

supported_output_ids = set(
    pred_values.columns[pred_values.notna().any(axis=0)].astype(str)
)

supported_tracks = {
    track
    for track, qnorm_id in track_to_first_id.items()
    if str(qnorm_id) in supported_output_ids
}
supported_qnorm_ids = {
    str(qnorm_id)
    for track in supported_tracks
    for qnorm_id in track_to_all_ids.get(track, [])
}

print(f"Requested tracks: {len(track_to_first_id)}")
print(f"Usable AlphaGenome tracks: {len(supported_tracks)}")
print(f"Ground-truth qnorm sample IDs: {len(supported_qnorm_ids)}")